# Experiments — LR vs DistilBERT

Thin notebook that calls the `src/` modules to train, evaluate, and compare both models.

To run from scratch: clear all outputs, then run all cells top-to-bottom.

In [ ]:
import sys
sys.path.insert(0, '..')  # so we can import from src/

import yaml
import pandas as pd

from src.preprocessing import preprocess_dataframe, download_nltk_resources
from src.utils import set_seed, ensure_dirs
from src import evaluate

download_nltk_resources()

## Load and preprocess data

In [ ]:
train_raw = pd.read_csv('../data/raw/train.csv')
test_raw  = pd.read_csv('../data/raw/test.csv')

print(f"Train: {train_raw.shape}  |  Test: {test_raw.shape}")

train_df = preprocess_dataframe(train_raw)
test_df  = preprocess_dataframe(test_raw)

train_df.head(3)

## Logistic Regression

In [ ]:
with open('../config/lr_config.yaml') as f:
    lr_cfg = yaml.safe_load(f)

set_seed(lr_cfg['data']['random_state'])
ensure_dirs(lr_cfg['output']['model_dir'], lr_cfg['output']['figures_dir'],
            '../outputs/submissions')

from src.models.lr_model import LogisticRegressionModel

lr_model = LogisticRegressionModel(lr_cfg)
y_true_lr, y_pred_lr = lr_model.fit(train_df)

print("LR metrics:", lr_model.val_metrics)

In [ ]:
evaluate.plot_confusion_matrix(y_true_lr, y_pred_lr, title='LR — Confusion Matrix')
evaluate.plot_feature_importance(
    lr_model.get_feature_names(),
    lr_model.get_coefficients(),
    top_n=20,
    title='LR — Top 20 Features'
)

In [ ]:
lr_model.save('../' + lr_cfg['output']['model_dir'])

## DistilBERT

> Training takes several minutes — GPU recommended. Switch `epochs` in `config/bert_config.yaml` for quick tests.

In [ ]:
with open('../config/bert_config.yaml') as f:
    bert_cfg = yaml.safe_load(f)

set_seed(bert_cfg['data']['random_state'])
ensure_dirs(bert_cfg['output']['model_dir'], bert_cfg['output']['figures_dir'])

from src.models.bert_model import DistilBertClassifier

bert_model = DistilBertClassifier(bert_cfg)
y_true_bert, y_pred_bert = bert_model.fit(train_df)

print("BERT metrics:", bert_model.val_metrics)

In [ ]:
evaluate.plot_confusion_matrix(y_true_bert, y_pred_bert, title='DistilBERT — Confusion Matrix')

In [ ]:
bert_model.save('../' + bert_cfg['output']['model_dir'])

## Side-by-side comparison

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {'model': 'Logistic Regression', **lr_model.val_metrics},
    {'model': 'DistilBERT',          **bert_model.val_metrics},
]).set_index('model')

comparison

## Generate test predictions

In [ ]:
# LR submission
lr_preds = lr_model.predict(test_df['text'])
lr_sub = pd.DataFrame({'id': test_df['id'], 'target': lr_preds})
lr_sub.to_csv('../outputs/submissions/submission_lr.csv', index=False)
print("LR submission saved")

# BERT submission
bert_preds = bert_model.predict(test_df)
bert_sub = pd.DataFrame({'id': test_df['id'], 'target': bert_preds})
bert_sub.to_csv('../outputs/submissions/submission_bert.csv', index=False)
print("BERT submission saved")